# RAG Retrieval Evaluation: Ranking Metrics for Retrieved Chunks

## MSA 8700 — Module 7: RAG Evaluation

This notebook provides a comprehensive, step-by-step walkthrough of the evaluation metrics used to assess **ranked chunk retrieval** in Retrieval-Augmented Generation (RAG) systems.

Evaluating ranked retrieval in RAG systems borrows heavily from **Information Retrieval (IR)** theory but adapts it to the nuances of chunk-level relevance.

## What You Will Learn

| Category | Metrics | Key Question |
|----------|---------|-------------|
| **Precision-Oriented** | P@K, AP, MAP, R-Precision, AUC-PR | "Of the chunks we retrieved, how many were useful?" |
| **Recall-Oriented** | R@K, MR@K, Coverage Recall | "Of all useful chunks, how many did we find?" |
| **Rank-Sensitive** | DCG, NDCG, RR, MRR, ERR | "Are the best chunks ranked highest?" |
| **RAG-Specific** | Context Precision, Context Recall, Hit Rate, Chunk Attribution | "Does retrieval quality actually help generation?" |
| **Evaluation Protocol** | Nugget Recall, Metric@K Curves, Stratified Evaluation | "How do we run a rigorous evaluation?" |

---
## Core Concepts Before the Metrics

In ranked retrieval evaluation, you need three things:

1. **A query** — the user's question or information need
2. **A ranked list of retrieved chunks** — ordered by the retriever's confidence (index 0 = top result)
3. **Relevance judgments** — ground-truth labels for each chunk

### Binary vs. Graded Relevance

| Type | Values | Example | When to Use |
|------|--------|---------|-------------|
| **Binary** | relevant / not relevant | `{"c1", "c3", "c5"}` | Simple yes/no relevance |
| **Graded** | 0, 1, 2, ... | `{"c1": 2, "c2": 0, "c3": 1}` | Chunk partially answers the query |

The challenge in RAG is that relevance can be **partial** — a chunk may contain *part* of an answer — so binary judgments are often insufficient. Graded relevance captures this nuance.

### Data Conventions in This Notebook

- `retrieved`: list of chunk IDs in ranked order (index 0 = top result)
- `relevant`: set of chunk IDs that are ground-truth relevant
- `grades`: dict mapping chunk_id → relevance score (e.g., 0/1/2)
- `queries`: list of (retrieved, relevant) tuples for multi-query metrics

---
## Setup: Imports and Sample Data

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['font.size'] = 12

In [ ]:
# --- Single query example ---
RETRIEVED = ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c9", "c10"]
RELEVANT  = {"c1", "c3", "c5", "c8"}   # ground-truth relevant chunks

# Graded relevance: 0=not relevant, 1=partially relevant, 2=highly relevant
GRADES = {
    "c1": 2, "c2": 0, "c3": 1, "c4": 0,
    "c5": 2, "c6": 0, "c7": 0, "c8": 1,
    "c9": 0, "c10": 0,
}

# Multi-query dataset  (retrieved_list, relevant_set)
QUERIES = [
    (["c1", "c2", "c3", "c4", "c5"],          {"c1", "c3"}),
    (["c6", "c7", "c1", "c8", "c9"],          {"c6", "c8"}),
    (["c2", "c5", "c3", "c10", "c4"],         {"c3", "c10"}),
    (["c1", "c3", "c2", "c5", "c8"],          {"c1", "c3", "c5", "c8"}),
]

# Multi-query with graded relevance
QUERIES_GRADED = [
    (["c1", "c2", "c3", "c4", "c5"], {"c1": 2, "c2": 0, "c3": 1, "c4": 0, "c5": 2}),
    (["c6", "c7", "c1", "c8", "c9"], {"c6": 2, "c7": 0, "c1": 0, "c8": 1, "c9": 0}),
]

print("Sample Data:")
print(f"  Retrieved : {RETRIEVED}")
print(f"  Relevant  : {sorted(RELEVANT)}")
print(f"  Grades    : {GRADES}")
print(f"  # Queries : {len(QUERIES)}")

Let's visualize our sample data to build intuition. In the ranked list below, **relevant** chunks are highlighted.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2))
for i, chunk in enumerate(RETRIEVED):
    color = '#2ecc71' if chunk in RELEVANT else '#e74c3c'
    label = f"{chunk}\n(grade={GRADES[chunk]})"
    ax.barh(0, 1, left=i, color=color, edgecolor='white', linewidth=2, height=0.6)
    ax.text(i + 0.5, 0, label, ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 10)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel('Rank Position (0 = top result)', fontsize=12)
ax.set_yticks([])
ax.set_title('Ranked Retrieved Chunks — Green = Relevant, Red = Not Relevant', fontsize=13)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Relevant'),
                   Patch(facecolor='#e74c3c', label='Not Relevant')]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

---
# 1. Precision-Oriented Metrics

Precision metrics answer: **"Of the chunks we retrieved, how many were actually useful?"**

These metrics focus on the quality of the retrieved set — they penalize the retriever for returning irrelevant chunks.

## 1.1 Precision@K (P@K)

**P@K** is the fraction of the top-K retrieved chunks that are relevant.

$$P@K = \frac{|\{\text{relevant chunks in top-K}\}|}{K}$$

**Key properties:**
- Answers: "Of the first K results, how many were useful?"
- Ignores rank order *within* K — treats position 1 and position K equally
- Sensitive to your choice of K
- Common choices: K ∈ {1, 3, 5, 10}

**Example:**
```
retrieved = ["c1", "c2", "c3", "c4", "c5"]
relevant  = {"c1", "c3"}
P@3 = 2/3 ≈ 0.667   (c1 and c3 are relevant; c2 is not)
P@5 = 2/5 = 0.4
```

In [ ]:
def precision_at_k(retrieved, relevant, k):
    """
    P@K: fraction of top-K retrieved chunks that are relevant.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer")
    top_k = retrieved[:k]
    hits  = sum(1 for chunk in top_k if chunk in relevant)
    return hits / k

In [ ]:
print("Step-by-step P@K Calculation")
print(f"Retrieved: {RETRIEVED}")
print(f"Relevant:  {sorted(RELEVANT)}")
print()

for k in [1, 3, 5, 10]:
    top_k = RETRIEVED[:k]
    hits = [c for c in top_k if c in RELEVANT]
    p_at_k = precision_at_k(RETRIEVED, RELEVANT, k)
    print(f"P@{k:<2} : top-{k} = {top_k}")
    print(f"       hits = {hits} ({len(hits)} relevant)")
    print(f"       P@{k} = {len(hits)}/{k} = {p_at_k:.4f}")
    print()

## 1.2 Average Precision (AP)

**AP** addresses the rank-insensitivity of P@K. It computes P@K at each position where a relevant chunk appears, then averages those values.

$$AP = \frac{1}{|R|} \sum_{k=1}^{n} P@k \cdot \mathbb{1}[\text{retrieved}[k] \in R]$$

where $R$ is the set of relevant chunks and $\mathbb{1}[\cdot]$ is 1 if the chunk at rank $k$ is relevant.

**Key properties:**
- Heavily rewards systems that place relevant chunks near the top
- Penalizes gaps between relevant results
- Single number that summarizes the full precision-recall curve

**Example:**
```
retrieved = ["c1", "c2", "c3", "c4", "c5"]
relevant  = {"c1", "c3"}
Relevant hits at ranks 1 and 3:
  P@1 = 1/1 = 1.0
  P@3 = 2/3 ≈ 0.667
AP = (1.0 + 0.667) / 2 ≈ 0.833
```

In [ ]:
def average_precision(retrieved, relevant):
    """
    AP: mean of P@k values computed at each rank where a relevant chunk appears.
    """
    if not relevant:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for rank, chunk in enumerate(retrieved, start=1):
        if chunk in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / len(relevant)

In [ ]:
print("Step-by-step Average Precision Calculation")
print(f"Retrieved: {RETRIEVED}")
print(f"Relevant:  {sorted(RELEVANT)}")
print()

hits = 0
precision_sum = 0.0
print(f"{'Rank':<6} {'Chunk':<8} {'Relevant?':<12} {'Hits':<6} {'P@k':<10} {'Included?'}")
print("-" * 60)

for rank, chunk in enumerate(RETRIEVED, start=1):
    is_rel = chunk in RELEVANT
    if is_rel:
        hits += 1
        p_at_k = hits / rank
        precision_sum += p_at_k
        print(f"{rank:<6} {chunk:<8} {'YES':<12} {hits:<6} {p_at_k:<10.4f} {'<-- included in AP sum'}")
    else:
        p_at_k = hits / rank
        print(f"{rank:<6} {chunk:<8} {'no':<12} {hits:<6} {p_at_k:<10.4f} {''}")

ap = precision_sum / len(RELEVANT)
print(f"\nAP = sum of included P@k / |relevant|")
print(f"   = {precision_sum:.4f} / {len(RELEVANT)}")
print(f"   = {ap:.4f}")

### Why AP matters: comparing two rankings

Let's see how AP distinguishes between a **good** ranking (relevant chunks at top) and a **bad** ranking (relevant chunks buried).

In [ ]:
good_ranking = ["c1", "c3", "c5", "c8", "c2", "c4", "c6", "c7", "c9", "c10"]
bad_ranking  = ["c2", "c4", "c6", "c7", "c9", "c10", "c1", "c3", "c5", "c8"]

ap_good = average_precision(good_ranking, RELEVANT)
ap_bad  = average_precision(bad_ranking, RELEVANT)
ap_orig = average_precision(RETRIEVED, RELEVANT)

print(f"Good ranking (all relevant first): {good_ranking[:6]}...")
print(f"  AP = {ap_good:.4f}")
print(f"\nOriginal ranking:                 {RETRIEVED[:6]}...")
print(f"  AP = {ap_orig:.4f}")
print(f"\nBad ranking (all relevant last):   {bad_ranking[:6]}...")
print(f"  AP = {ap_bad:.4f}")
print(f"\n--> AP clearly distinguishes ranking quality!")

## 1.3 Mean Average Precision (MAP)

**MAP** is the mean of AP scores across multiple queries. It is the standard aggregate metric for comparing retrieval systems.

$$MAP = \frac{1}{|Q|} \sum_{q=1}^{|Q|} AP(q)$$

In [ ]:
def mean_average_precision(queries):
    """MAP: mean of AP scores across multiple queries."""
    ap_scores = [average_precision(r, rel) for r, rel in queries]
    return float(np.mean(ap_scores))

In [ ]:
print("MAP Calculation Across 4 Queries")
print("=" * 50)

ap_scores = []
for i, (ret, rel) in enumerate(QUERIES):
    ap = average_precision(ret, rel)
    ap_scores.append(ap)
    print(f"Query {i+1}: retrieved={ret}")
    print(f"          relevant ={sorted(rel)}")
    print(f"          AP = {ap:.4f}")
    print()

map_score = np.mean(ap_scores)
print(f"MAP = mean({[f'{a:.4f}' for a in ap_scores]})")
print(f"    = {sum(ap_scores):.4f} / {len(ap_scores)}")
print(f"    = {map_score:.4f}")

## 1.4 R-Precision

**R-Precision** computes precision at exactly rank R, where R = |relevant| (the total number of relevant chunks for that query).

$$\text{R-Precision} = P@R \quad \text{where } R = |\text{relevant}|$$

**Key property:** Self-normalizes across queries with different numbers of relevant chunks. If a query has 3 relevant chunks, we check whether the top 3 results contain those 3 chunks.

In [ ]:
def r_precision(retrieved, relevant):
    """R-Precision: precision at rank R, where R = |relevant|."""
    r = len(relevant)
    if r == 0:
        return 0.0
    return precision_at_k(retrieved, relevant, k=r)

In [ ]:
r = len(RELEVANT)
top_r = RETRIEVED[:r]
hits = [c for c in top_r if c in RELEVANT]

print(f"Relevant set: {sorted(RELEVANT)} (R = {r})")
print(f"Top-{r} retrieved: {top_r}")
print(f"Hits in top-{r}: {hits} ({len(hits)} relevant)")
print(f"R-Precision = {len(hits)}/{r} = {r_precision(RETRIEVED, RELEVANT):.4f}")

## 1.5 Interpolated Precision-Recall Curve & AUC-PR

The **interpolated precision-recall curve** plots precision at standard recall breakpoints (0.0, 0.1, ..., 1.0).

**Interpolation rule:** At each recall level $r$, the interpolated precision is the maximum precision at any recall level $r' \geq r$:

$$P_{\text{interp}}(r) = \max_{r' \geq r} P(r')$$

The **AUC-PR** (Area Under the Precision-Recall curve) summarizes the entire curve into a single number using the trapezoidal rule.

In [ ]:
def interpolated_precision_recall_curve(retrieved, relevant, recall_levels=None):
    """
    Computes interpolated precision at standard recall breakpoints.
    Interpolation: P(r) = max(P(r') for r' >= r)
    """
    if recall_levels is None:
        recall_levels = [i / 10 for i in range(11)]

    total_relevant = len(relevant)
    if total_relevant == 0:
        return {r: 0.0 for r in recall_levels}

    # Build raw (recall, precision) pairs at each rank
    raw_points = []
    hits = 0
    for rank, chunk in enumerate(retrieved, start=1):
        if chunk in relevant:
            hits += 1
            rec  = hits / total_relevant
            prec = hits / rank
            raw_points.append((rec, prec))

    # Interpolate: for each level, take max precision at recall >= level
    interpolated = {}
    for level in recall_levels:
        candidates = [p for (r, p) in raw_points if r >= level]
        interpolated[level] = max(candidates) if candidates else 0.0
    return interpolated


def auc_pr(retrieved, relevant):
    """Area under the Precision-Recall curve using the trapezoidal rule."""
    curve = interpolated_precision_recall_curve(retrieved, relevant)
    levels = sorted(curve.keys())
    precisions = [curve[l] for l in levels]
    return float(np.trapz(precisions, levels))

In [ ]:
print("Step-by-step Precision-Recall Curve Construction")
print(f"Retrieved: {RETRIEVED}")
print(f"Relevant:  {sorted(RELEVANT)} (total = {len(RELEVANT)})")
print()

print("Raw (recall, precision) points at each relevant hit:")
hits = 0
for rank, chunk in enumerate(RETRIEVED, start=1):
    if chunk in RELEVANT:
        hits += 1
        rec = hits / len(RELEVANT)
        prec = hits / rank
        print(f"  Rank {rank}: {chunk} is relevant -> hits={hits}, recall={rec:.2f}, precision={prec:.4f}")

print("\nInterpolated Precision-Recall Curve:")
curve = interpolated_precision_recall_curve(RETRIEVED, RELEVANT)
for level in sorted(curve):
    bar = '█' * int(curve[level] * 30)
    print(f"  Recall={level:.1f}  Precision={curve[level]:.3f}  {bar}")

print(f"\nAUC-PR = {auc_pr(RETRIEVED, RELEVANT):.4f}")

In [ ]:
# Plot the interpolated precision-recall curve
curve = interpolated_precision_recall_curve(RETRIEVED, RELEVANT)
recall_vals = sorted(curve.keys())
precision_vals = [curve[r] for r in recall_vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.step(recall_vals, precision_vals, where='post', linewidth=2, color='#3498db')
ax.fill_between(recall_vals, precision_vals, step='post', alpha=0.2, color='#3498db')
ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Interpolated Precision', fontsize=13)
ax.set_title(f'Interpolated Precision-Recall Curve (AUC-PR = {auc_pr(RETRIEVED, RELEVANT):.4f})', fontsize=14)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 2. Recall-Oriented Metrics

Recall metrics answer: **"Of all the relevant chunks that exist, how many did we actually retrieve?"**

In RAG, recall is arguably **more important** than precision because a missed relevant chunk can mean a missed fact, leading to **hallucination or incomplete answers**.

## 2.1 Recall@K (R@K)

**R@K** is the fraction of all relevant chunks that appear in the top-K results.

$$R@K = \frac{|\{\text{relevant chunks in top-K}\}|}{|\text{all relevant chunks}|}$$

**Key difference from P@K:**
- P@K divides by K (the retrieval cutoff)
- R@K divides by |relevant| (the total number of relevant chunks)

**In RAG, this is critical** — the denominator (total relevant chunks) must be known, which requires thorough relevance annotation.

In [ ]:
def recall_at_k(retrieved, relevant, k):
    """R@K: fraction of all relevant chunks that appear in the top-K results."""
    if not relevant:
        return 0.0
    top_k = set(retrieved[:k])
    hits  = len(top_k & relevant)
    return hits / len(relevant)


def mean_recall_at_k(queries, k):
    """MR@K: average Recall@K across multiple queries."""
    scores = [recall_at_k(r, rel, k) for r, rel in queries]
    return float(np.mean(scores))

In [ ]:
print("Step-by-step Recall@K Calculation")
print(f"Retrieved: {RETRIEVED}")
print(f"Relevant:  {sorted(RELEVANT)} (total = {len(RELEVANT)})")
print()

for k in [1, 3, 5, 10]:
    top_k = set(RETRIEVED[:k])
    hits = top_k & RELEVANT
    r_at_k = recall_at_k(RETRIEVED, RELEVANT, k)
    print(f"R@{k:<2}: top-{k} = {RETRIEVED[:k]}")
    print(f"       hits = {sorted(hits)} ({len(hits)} found out of {len(RELEVANT)})")
    print(f"       R@{k} = {len(hits)}/{len(RELEVANT)} = {r_at_k:.4f}")
    print()

### Precision vs. Recall: Side-by-Side Comparison

Let's plot P@K and R@K together to see how they behave as K increases.

In [ ]:
k_values = list(range(1, len(RETRIEVED) + 1))
p_vals = [precision_at_k(RETRIEVED, RELEVANT, k) for k in k_values]
r_vals = [recall_at_k(RETRIEVED, RELEVANT, k) for k in k_values]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, p_vals, 'o-', label='Precision@K', color='#e74c3c', linewidth=2, markersize=8)
ax.plot(k_values, r_vals, 's-', label='Recall@K', color='#3498db', linewidth=2, markersize=8)
ax.set_xlabel('K (cutoff rank)', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Precision@K vs Recall@K — The Classic Trade-Off', fontsize=14)
ax.legend(fontsize=12)
ax.set_xticks(k_values)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Annotate the trade-off
ax.annotate('Precision tends to\ndecrease with K', xy=(8, p_vals[7]),
            fontsize=10, ha='center', color='#e74c3c')
ax.annotate('Recall increases\nwith K', xy=(8, r_vals[7]),
            fontsize=10, ha='center', color='#3498db')

plt.tight_layout()
plt.show()

## 2.2 Coverage Recall

**Coverage Recall** is a RAG-specific variant that measures whether the retrieved set collectively *covers* all **sub-questions or facets** of a complex query.

$$\text{Coverage Recall} = \frac{|\{\text{facets with at least one retrieved chunk}\}|}{|\text{all facets}|}$$

This is essential for **multi-hop or multi-faceted queries** where a single chunk may not address all aspects of the question.

In [ ]:
def coverage_recall(retrieved, facets):
    """
    Coverage Recall: fraction of query facets covered by
    at least one retrieved chunk.
    """
    if not facets:
        return 0.0
    retrieved_set = set(retrieved)
    covered = sum(1 for facet in facets if retrieved_set & facet)
    return covered / len(facets)

In [ ]:
retrieved_subset = RETRIEVED[:5]  # Only using top-5
facets = [
    {"c1", "c2"},    # facet 1: construction details
    {"c3", "c5"},    # facet 2: dimensions
    {"c8", "c9"},    # facet 3: historical context
    {"c6", "c7"},    # facet 4: location info
]

print(f"Retrieved (top-5): {retrieved_subset}")
print(f"Retrieved set: {set(retrieved_subset)}")
print()

for i, facet in enumerate(facets):
    overlap = set(retrieved_subset) & facet
    status = '✓ COVERED' if overlap else '✗ NOT COVERED'
    print(f"  Facet {i+1}: {sorted(facet)} -> overlap = {sorted(overlap)} -> {status}")

cr = coverage_recall(retrieved_subset, facets)
covered_count = sum(1 for f in facets if set(retrieved_subset) & f)
print(f"\nCoverage Recall = {covered_count}/{len(facets)} = {cr:.4f}")

---
# 3. Rank-Sensitive Combined Metrics

These metrics go beyond simple set-based comparisons. They account for **where** in the ranked list relevant chunks appear, and they can handle **graded relevance** (not just binary).

This is particularly well-suited to RAG because chunk relevance is rarely binary — a chunk may be *partially* relevant.

## 3.1 DCG@K (Discounted Cumulative Gain)

**DCG** uses graded relevance and applies a **logarithmic discount** to lower-ranked positions. Higher-ranked relevant chunks contribute more.

$$DCG@K = \sum_{i=1}^{K} \frac{\text{grade}_i}{\log_2(i + 1)}$$

The logarithmic discount means:
- Rank 1: divide by $\log_2(2) = 1.0$ (full credit)
- Rank 2: divide by $\log_2(3) \approx 1.585$ (63% credit)
- Rank 5: divide by $\log_2(6) \approx 2.585$ (39% credit)
- Rank 10: divide by $\log_2(11) \approx 3.459$ (29% credit)

In [ ]:
def dcg_at_k(retrieved, grades, k, log_base=2):
    """DCG@K: Discounted Cumulative Gain."""
    dcg = 0.0
    for rank, chunk in enumerate(retrieved[:k], start=1):
        grade = grades.get(chunk, 0)
        dcg += grade / math.log(rank + 1, log_base)
    return dcg

In [ ]:
print("Step-by-step DCG@5 Calculation")
print(f"Retrieved: {RETRIEVED[:5]}")
print(f"Grades:    {GRADES}")
print()

k = 5
dcg = 0.0
print(f"{'Rank':<6} {'Chunk':<8} {'Grade':<8} {'log2(r+1)':<12} {'Gain':<10} {'Cumulative DCG'}")
print("-" * 65)

for rank, chunk in enumerate(RETRIEVED[:k], start=1):
    grade = GRADES.get(chunk, 0)
    discount = math.log(rank + 1, 2)
    gain = grade / discount
    dcg += gain
    print(f"{rank:<6} {chunk:<8} {grade:<8} {discount:<12.4f} {gain:<10.4f} {dcg:.4f}")

print(f"\nDCG@{k} = {dcg:.4f}")

## 3.2 NDCG@K (Normalized DCG)

**NDCG** is arguably the **gold standard** for ranked evaluation. It normalizes DCG by the **Ideal DCG (IDCG)** — the DCG achieved by the perfect ranking.

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

where IDCG@K is computed by sorting all chunks by relevance grade in descending order.

- **NDCG = 1.0** means the ranking is perfect (most relevant items first)
- **NDCG = 0.0** means no relevant items were retrieved

In [ ]:
def ndcg_at_k(retrieved, grades, k, log_base=2):
    """NDCG@K: Normalized DCG — DCG@K / IDCG@K."""
    actual_dcg = dcg_at_k(retrieved, grades, k, log_base)
    ideal_ranking = sorted(grades.keys(), key=lambda c: grades[c], reverse=True)
    ideal_dcg = dcg_at_k(ideal_ranking, grades, k, log_base)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg


def mean_ndcg_at_k(queries, k):
    """Mean NDCG@K across multiple queries."""
    scores = [ndcg_at_k(r, g, k) for r, g in queries]
    return float(np.mean(scores))

In [ ]:
k = 5
print(f"NDCG@{k} Calculation")
print("=" * 50)

# Actual DCG
actual = dcg_at_k(RETRIEVED, GRADES, k)
print(f"\nActual ranking: {RETRIEVED[:k]}")
print(f"Grades:         {[GRADES[c] for c in RETRIEVED[:k]]}")
print(f"DCG@{k} = {actual:.4f}")

# Ideal DCG
ideal_ranking = sorted(GRADES.keys(), key=lambda c: GRADES[c], reverse=True)
ideal = dcg_at_k(ideal_ranking, GRADES, k)
print(f"\nIdeal ranking:  {ideal_ranking[:k]}")
print(f"Grades:         {[GRADES[c] for c in ideal_ranking[:k]]}")
print(f"IDCG@{k} = {ideal:.4f}")

ndcg = actual / ideal if ideal > 0 else 0.0
print(f"\nNDCG@{k} = DCG / IDCG = {actual:.4f} / {ideal:.4f} = {ndcg:.4f}")
print(f"\nInterpretation: The ranking achieves {ndcg*100:.1f}% of the ideal DCG.")

In [ ]:
# Visualize actual vs ideal ranking
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

for ax, ranking, title in [
    (axes[0], RETRIEVED[:5], f'Actual Ranking (DCG={actual:.3f})'),
    (axes[1], ideal_ranking[:5], f'Ideal Ranking (IDCG={ideal:.3f})'),
]:
    for i, chunk in enumerate(ranking):
        grade = GRADES.get(chunk, 0)
        colors = {0: '#e74c3c', 1: '#f39c12', 2: '#2ecc71'}
        ax.barh(0, 1, left=i, color=colors[grade], edgecolor='white', linewidth=2, height=0.6)
        ax.text(i + 0.5, 0, f"{chunk}\n(g={grade})", ha='center', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 5)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Rank Position')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Grade 2 (highly relevant)'),
                   Patch(facecolor='#f39c12', label='Grade 1 (partially)'),
                   Patch(facecolor='#e74c3c', label='Grade 0 (not relevant)')]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 1.12))
plt.tight_layout()
plt.show()

## 3.3 Reciprocal Rank (RR) & Mean Reciprocal Rank (MRR)

**RR** is simply $1 / \text{rank}$ of the **first** relevant chunk found.

$$RR = \frac{1}{\text{rank of first relevant chunk}}$$

**MRR** averages RR across multiple queries.

$$MRR = \frac{1}{|Q|} \sum_{q=1}^{|Q|} \frac{1}{\text{rank}_q}$$

**Answers:** "On average, how quickly does the system surface *at least one* relevant chunk?"

**Limitation:** Only considers the first hit — ignores the rest of the ranked list.

In [ ]:
def reciprocal_rank(retrieved, relevant):
    """RR: 1 / rank of the first relevant chunk found."""
    for rank, chunk in enumerate(retrieved, start=1):
        if chunk in relevant:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank(queries):
    """MRR: Mean Reciprocal Rank across multiple queries."""
    rr_scores = [reciprocal_rank(r, rel) for r, rel in queries]
    return float(np.mean(rr_scores))

In [ ]:
print("Reciprocal Rank Examples")
print("=" * 50)

# Single query
print(f"\nSingle Query:")
print(f"  Retrieved: {RETRIEVED}")
print(f"  Relevant:  {sorted(RELEVANT)}")
for rank, chunk in enumerate(RETRIEVED, start=1):
    if chunk in RELEVANT:
        print(f"  First relevant chunk: {chunk} at rank {rank}")
        print(f"  RR = 1/{rank} = {1/rank:.4f}")
        break

# Multiple queries
print(f"\nMRR Across 4 Queries:")
rr_scores = []
for i, (ret, rel) in enumerate(QUERIES):
    rr = reciprocal_rank(ret, rel)
    rr_scores.append(rr)
    # Find first hit
    first_rank = next((r for r, c in enumerate(ret, 1) if c in rel), 'none')
    print(f"  Query {i+1}: first relevant at rank {first_rank} -> RR = {rr:.4f}")

mrr = mean_reciprocal_rank(QUERIES)
print(f"\n  MRR = mean({[f'{s:.4f}' for s in rr_scores]}) = {mrr:.4f}")

## 3.4 Expected Reciprocal Rank (ERR)

**ERR** extends MRR by modeling the probability that a user is **satisfied** at each rank position, incorporating **graded relevance**.

It accounts for the **cascading effect** of relevance — if a highly relevant chunk is found early, the user is less likely to continue scanning.

$$ERR = \sum_{r=1}^{n} \frac{1}{r} \cdot P(\text{satisfied at rank } r)$$

where:
$$P(\text{satisfied at rank } r) = R_r \cdot \prod_{i=1}^{r-1} (1 - R_i)$$

and $R_i = \text{grade}_i / \text{max\_grade}$ is the probability of satisfaction at rank $i$.

In [ ]:
def expected_reciprocal_rank(retrieved, grades, max_grade=2):
    """ERR: Expected Reciprocal Rank with graded relevance."""
    err = 0.0
    p_not_satisfied = 1.0
    for rank, chunk in enumerate(retrieved, start=1):
        grade = grades.get(chunk, 0)
        rel_prob = grade / max_grade
        err += p_not_satisfied * rel_prob * (1.0 / rank)
        p_not_satisfied *= (1.0 - rel_prob)
        if p_not_satisfied < 1e-10:
            break
    return err

In [ ]:
print("Step-by-step ERR Calculation")
print(f"Retrieved: {RETRIEVED}")
print(f"Grades:    {GRADES}")
print(f"Max grade: 2")
print()

err = 0.0
p_not_sat = 1.0
print(f"{'Rank':<6} {'Chunk':<8} {'Grade':<8} {'R_i':<8} {'P(not sat)':<14} {'Contribution':<14} {'Cumulative ERR'}")
print("-" * 80)

for rank, chunk in enumerate(RETRIEVED, start=1):
    grade = GRADES.get(chunk, 0)
    rel_prob = grade / 2
    contribution = p_not_sat * rel_prob * (1.0 / rank)
    err += contribution
    print(f"{rank:<6} {chunk:<8} {grade:<8} {rel_prob:<8.2f} {p_not_sat:<14.4f} {contribution:<14.4f} {err:.4f}")
    p_not_sat *= (1.0 - rel_prob)
    if p_not_sat < 1e-10:
        print("  (user almost certainly satisfied, stopping)")
        break

print(f"\nERR = {err:.4f}")
print(f"\nInterpretation: ERR models a user scanning top-down.")
print(f"A highly relevant chunk (grade=2) at rank 1 gives high satisfaction probability,")
print(f"reducing the contribution of later results.")

---
# 4. RAG-Specific Adapted Metrics

These metrics were developed specifically for or adapted to the RAG paradigm. They bridge **retrieval quality** with **generation quality**.

## 4.1 Context Precision (RAGAS-style)

**Context Precision** measures whether retrieved chunks that ARE relevant appear **earlier** in the ranked list than those that are NOT.

It is computed as **Average Precision** — equivalent to MAP for a single query. Popularized by the [RAGAS](https://docs.ragas.io/) evaluation framework.

$$\text{Context Precision} = AP(\text{retrieved}, \text{relevant})$$

In [ ]:
def context_precision(retrieved, relevant):
    """Context Precision (RAGAS-style): AP over the ranked list."""
    return average_precision(retrieved, relevant)

In [ ]:
print("Context Precision: Comparing Two Rankings")
print("=" * 55)

good_order = ["c1", "c2", "c3", "c4", "c5"]
bad_order  = ["c2", "c4", "c1", "c3", "c5"]
relevant_sub = {"c1", "c3"}

cp_good = context_precision(good_order, relevant_sub)
cp_bad  = context_precision(bad_order, relevant_sub)

print(f"\nGood ranking: {good_order}  (relevant = {sorted(relevant_sub)})")
print(f"  c1 at rank 1, c3 at rank 3")
print(f"  AP = (P@1 + P@3) / 2 = (1.0 + 0.667) / 2 = {cp_good:.4f}")

print(f"\nBad ranking:  {bad_order}  (relevant = {sorted(relevant_sub)})")
print(f"  c1 at rank 3, c3 at rank 4")
print(f"  AP = (P@3 + P@4) / 2 = (0.333 + 0.5) / 2 = {cp_bad:.4f}")

print(f"\n--> Context Precision clearly rewards placing relevant chunks earlier!")

## 4.2 Context Recall (RAGAS-style, LLM-based)

**Context Recall** measures how much of the **ground-truth answer** can be attributed to the retrieved context. Unlike traditional recall (which counts chunks), this uses an **LLM judge** to check whether each claim in a reference answer is supported by at least one retrieved chunk.

$$\text{Context Recall} = \frac{|\{\text{claims supported by retrieved context}\}|}{|\text{all reference claims}|}$$

This is a **semantic, generation-aware recall** rather than a pure retrieval metric.

In [ ]:
def context_recall_llm_based(retrieved_texts, reference_claims, claim_supported):
    """
    Context Recall (RAGAS-style): fraction of reference answer claims
    supported by at least one retrieved chunk.
    Takes pre-computed LLM judge labels as input.
    """
    if not reference_claims:
        return 0.0
    if len(reference_claims) != len(claim_supported):
        raise ValueError("reference_claims and claim_supported must be the same length")
    return sum(claim_supported) / len(reference_claims)

In [ ]:
print("Context Recall (LLM-based) Example")
print("=" * 55)
print()

# Simulated LLM judge output
claims = [
    "The Eiffel Tower was built in 1889.",
    "It is located in Paris.",
    "It is 330 meters tall.",
]
supported = [True, True, False]

print("Reference claims and LLM judge results:")
for claim, sup in zip(claims, supported):
    status = '✓ Supported' if sup else '✗ NOT Supported'
    print(f"  [{status}] \"{claim}\"")

cr = context_recall_llm_based(RETRIEVED[:3], claims, supported)
print(f"\nContext Recall = {sum(supported)}/{len(claims)} = {cr:.4f}")
print(f"\nNote: In practice, an LLM (e.g., GPT-4 or Claude) judges each claim")
print(f"against the retrieved chunk texts to determine 'supported'.")

## 4.3 Hit Rate @K

**Hit Rate @K** is the simplest retrieval metric: what fraction of queries have **at least one** relevant chunk in the top-K?

$$\text{Hit Rate@K} = \frac{|\{\text{queries with } \geq 1 \text{ hit in top-K}\}|}{|\text{all queries}|}$$

It's coarse but practical — especially when you only need **one good chunk** to answer a question. Useful as a baseline or sanity check.

In [ ]:
def hit_rate_at_k(queries, k):
    """Hit Rate @K: fraction of queries with at least one hit in top-K."""
    hits = sum(
        1 for retrieved, relevant in queries
        if set(retrieved[:k]) & relevant
    )
    return hits / len(queries)

In [ ]:
print("Hit Rate @K Examples")
print("=" * 55)

for k in [1, 3, 5]:
    print(f"\nHit Rate @{k}:")
    hit_count = 0
    for i, (ret, rel) in enumerate(QUERIES):
        has_hit = bool(set(ret[:k]) & rel)
        hit_count += has_hit
        status = '✓ HIT' if has_hit else '✗ MISS'
        print(f"  Query {i+1}: top-{k} = {ret[:k]} | relevant = {sorted(rel)} -> {status}")
    hr = hit_rate_at_k(QUERIES, k)
    print(f"  Hit Rate@{k} = {hit_count}/{len(QUERIES)} = {hr:.4f}")

## 4.4 Chunk Attribution Rate

**Chunk Attribution Rate** measures what fraction of retrieved chunks are actually **cited or used** by the generator in its final answer.

$$\text{Attribution Rate} = \frac{|\{\text{retrieved chunks cited in answer}\}|}{|\text{retrieved chunks}|}$$

This bridges retrieval quality with generation quality. **Low attribution** suggests the retriever is fetching technically relevant but practically ignored chunks — wasting context window space.

In [ ]:
def chunk_attribution_rate(retrieved, cited_chunks):
    """Fraction of retrieved chunks that were cited by the generator."""
    if not retrieved:
        return 0.0
    cited = sum(1 for c in retrieved if c in cited_chunks)
    return cited / len(retrieved)

In [ ]:
cited = {"c1", "c3", "c5"}

print("Chunk Attribution Rate")
print("=" * 55)
print(f"Retrieved chunks: {RETRIEVED}")
print(f"Cited by generator: {sorted(cited)}")
print()

for chunk in RETRIEVED:
    status = '✓ CITED' if chunk in cited else '  not cited'
    print(f"  {chunk}: {status}")

car = chunk_attribution_rate(RETRIEVED, cited)
print(f"\nAttribution Rate = {len(cited & set(RETRIEVED))}/{len(RETRIEVED)} = {car:.4f}")
print(f"\nInterpretation: Only {car*100:.0f}% of retrieved chunks were actually used.")
print(f"The rest consumed context window space without contributing to the answer.")

---
# 5. Evaluation Protocol Utilities

Beyond individual metrics, a rigorous RAG evaluation requires careful protocol design:

- **Relevance annotation depth** matters enormously. Shallow pools (judging only top-10) can underestimate recall.
- **Nugget-based evaluation** decomposes a reference answer into atomic facts for fine-grained assessment.
- **Stratified evaluation** breaks queries into categories to reveal per-category strengths/weaknesses.
- **Sensitivity analysis across K** reveals whether the ranker degrades gracefully.

## 5.1 Nugget-based Recall

**Nugget-based evaluation** decomposes a reference answer into **atomic facts ("nuggets")** and checks which nuggets are covered by retrieved chunks. This is more fine-grained than document-level judgments and handles multi-faceted queries better.

Each nugget is a `(fact_description, supporting_chunk_ids)` pair. A nugget is "covered" if at least one supporting chunk was retrieved.

In [ ]:
def nugget_recall(retrieved, nuggets):
    """
    Nugget-based Recall: fraction of atomic facts covered by retrieved chunks.
    Each nugget is a (fact_description, set_of_supporting_chunk_ids) pair.
    """
    if not nuggets:
        return 0.0
    retrieved_set = set(retrieved)
    covered = sum(1 for _, support in nuggets if retrieved_set & support)
    return covered / len(nuggets)

In [ ]:
nuggets = [
    ("tower height",        {"c1", "c2"}),
    ("construction year",   {"c3"}),
    ("architect name",      {"c5", "c6"}),
    ("location",            {"c8", "c10"}),
]

retrieved_top5 = RETRIEVED[:5]

print("Nugget Recall Example")
print("=" * 55)
print(f"Retrieved (top-5): {retrieved_top5}")
print()

for nugget_name, support in nuggets:
    overlap = set(retrieved_top5) & support
    status = '✓ COVERED' if overlap else '✗ NOT COVERED'
    print(f"  Nugget: '{nugget_name}'")
    print(f"    Supporting chunks: {sorted(support)}")
    print(f"    Retrieved overlap: {sorted(overlap) if overlap else 'none'}")
    print(f"    Status: {status}")
    print()

nr = nugget_recall(retrieved_top5, nuggets)
covered_count = sum(1 for _, s in nuggets if set(retrieved_top5) & s)
print(f"Nugget Recall = {covered_count}/{len(nuggets)} = {nr:.4f}")

## 5.2 Sensitivity Analysis: Metric@K Curves

**Sensitivity analysis across K** involves plotting your metric as K varies from 1 to your context window limit. The shape of this curve reveals whether your ranker degrades gracefully or drops off sharply, informing decisions about **how many chunks to pass to the LLM**.

In [ ]:
def metric_at_k_curve(retrieved, relevant, metric_fn, k_values=None):
    """Compute any @K metric across a range of K values."""
    if k_values is None:
        k_values = list(range(1, len(retrieved) + 1))
    return {k: metric_fn(retrieved, relevant, k) for k in k_values}

In [ ]:
p_curve = metric_at_k_curve(RETRIEVED, RELEVANT, precision_at_k)
r_curve = metric_at_k_curve(RETRIEVED, RELEVANT, recall_at_k)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision curve
ks = sorted(p_curve.keys())
axes[0].bar(ks, [p_curve[k] for k in ks], color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('K', fontsize=13)
axes[0].set_ylabel('Precision@K', fontsize=13)
axes[0].set_title('Precision@K Sensitivity', fontsize=14)
axes[0].set_xticks(ks)
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', alpha=0.3)

# Recall curve
axes[1].bar(ks, [r_curve[k] for k in ks], color='#3498db', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('K', fontsize=13)
axes[1].set_ylabel('Recall@K', fontsize=13)
axes[1].set_title('Recall@K Sensitivity', fontsize=14)
axes[1].set_xticks(ks)
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Sensitivity Analysis: How Metrics Change with K', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("The shape of these curves informs how many chunks to pass to the LLM:")
print("  - If recall plateaus at K=5, passing more chunks wastes context window space.")
print("  - If precision drops sharply, later chunks are mostly noise.")

## 5.3 Stratified Evaluation

**Stratified evaluation** breaks your query set into categories and reports metrics per stratum. A retriever might excel at simple factoid queries but fail badly on multi-hop ones — aggregate metrics can hide this.

Common strata:
- **Factoid** vs. **multi-hop** queries
- **Short** vs. **long** queries
- **Temporal** vs. **static** queries
- Domain-specific categories

In [ ]:
def stratified_evaluation(queries_by_type, k=5):
    """Compute MAP, MRR, MR@K, Hit Rate per query stratum."""
    results = {}
    for stratum, queries in queries_by_type.items():
        results[stratum] = {
            "MAP":              mean_average_precision(queries),
            "MRR":              mean_reciprocal_rank(queries),
            f"MR@{k}":          mean_recall_at_k(queries, k),
            f"HitRate@{k}":     hit_rate_at_k(queries, k),
        }
    return results

In [ ]:
queries_by_type = {
    "factoid":   QUERIES[:2],
    "multi_hop": QUERIES[2:],
}

strat = stratified_evaluation(queries_by_type, k=5)

print("Stratified Evaluation Results")
print("=" * 55)

for stratum, metrics in strat.items():
    print(f"\n  [{stratum.upper()}] ({len(queries_by_type[stratum])} queries)")
    print(f"  {'-' * 45}")
    for metric_name, score in metrics.items():
        bar = '█' * int(score * 20)
        print(f"    {metric_name:<15} {score:.4f}  {bar}")

In [ ]:
# Visualize stratified results
strata = list(strat.keys())
metrics_list = list(strat[strata[0]].keys())

x = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db', '#e74c3c']

for i, stratum in enumerate(strata):
    values = [strat[stratum][m] for m in metrics_list]
    ax.bar(x + i * width, values, width, label=stratum.capitalize(), color=colors[i], alpha=0.8)
    # Add value labels on bars
    for j, v in enumerate(values):
        ax.text(x[j] + i * width, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Stratified Evaluation: Factoid vs Multi-Hop Queries', fontsize=14)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
# 6. Complete Demo: All Metrics at a Glance

Let's run all metrics on our sample data and compare them side by side.

In [ ]:
print("=" * 60)
print("  COMPLETE METRIC SUMMARY")
print("=" * 60)
print(f"  Retrieved : {RETRIEVED}")
print(f"  Relevant  : {sorted(RELEVANT)}")
print(f"  Grades    : {GRADES}")

# Precision
print(f"\n{'─' * 60}")
print("  PRECISION-ORIENTED METRICS")
print(f"{'─' * 60}")
for k in [1, 3, 5, 10]:
    print(f"    Precision@{k:<2}                        {precision_at_k(RETRIEVED, RELEVANT, k):.4f}")
print(f"    Average Precision (AP)               {average_precision(RETRIEVED, RELEVANT):.4f}")
print(f"    MAP (4 queries)                      {mean_average_precision(QUERIES):.4f}")
print(f"    R-Precision                          {r_precision(RETRIEVED, RELEVANT):.4f}")
print(f"    AUC-PR                               {auc_pr(RETRIEVED, RELEVANT):.4f}")

# Recall
print(f"\n{'─' * 60}")
print("  RECALL-ORIENTED METRICS")
print(f"{'─' * 60}")
for k in [1, 3, 5, 10]:
    print(f"    Recall@{k:<2}                           {recall_at_k(RETRIEVED, RELEVANT, k):.4f}")
print(f"    Mean Recall@5 (4 queries)            {mean_recall_at_k(QUERIES, 5):.4f}")

facets = [{"c1", "c2"}, {"c3", "c5"}, {"c8", "c9"}, {"c6", "c7"}]
print(f"    Coverage Recall (4 facets)            {coverage_recall(RETRIEVED[:5], facets):.4f}")

# Rank-Sensitive
print(f"\n{'─' * 60}")
print("  RANK-SENSITIVE METRICS")
print(f"{'─' * 60}")
for k in [3, 5, 10]:
    print(f"    DCG@{k:<2}                              {dcg_at_k(RETRIEVED, GRADES, k):.4f}")
    print(f"    NDCG@{k:<2}                             {ndcg_at_k(RETRIEVED, GRADES, k):.4f}")
print(f"    Reciprocal Rank                      {reciprocal_rank(RETRIEVED, RELEVANT):.4f}")
print(f"    MRR (4 queries)                      {mean_reciprocal_rank(QUERIES):.4f}")
print(f"    ERR (graded)                         {expected_reciprocal_rank(RETRIEVED, GRADES, max_grade=2):.4f}")

# RAG-Specific
print(f"\n{'─' * 60}")
print("  RAG-SPECIFIC METRICS")
print(f"{'─' * 60}")
print(f"    Context Precision                    {context_precision(RETRIEVED, RELEVANT):.4f}")
print(f"    Hit Rate@3 (4 queries)               {hit_rate_at_k(QUERIES, 3):.4f}")
print(f"    Hit Rate@5 (4 queries)               {hit_rate_at_k(QUERIES, 5):.4f}")
print(f"    Chunk Attribution Rate               {chunk_attribution_rate(RETRIEVED, {'c1', 'c3', 'c5'}):.4f}")
print(f"    Context Recall (simulated)           {context_recall_llm_based(RETRIEVED[:3], claims, supported):.4f}")
print(f"\n{'=' * 60}")

---
# 7. Choosing the Right Metric

The right choice depends on your RAG use case:

| Use Case | Recommended Metrics | Rationale |
|----------|-------------------|-----------|
| **Missing chunks is catastrophic** (medical, legal QA) | **Recall@K**, **Context Recall** | Must find all relevant evidence |
| **Tight context window** (few chunks allowed) | **NDCG@K**, **P@K** with small K | Every slot must count |
| **Conversational assistant** (one good answer) | **MRR** | Just need one relevant chunk fast |
| **General benchmarking** with graded relevance | **NDCG** | Most principled overall choice |
| **Multi-faceted queries** | **Coverage Recall**, **Nugget Recall** | Must cover all aspects |
| **Debug retriever vs. generator** | **Chunk Attribution Rate** | Is the retriever helping the generator? |
| **Compare retrieval systems** | **MAP**, **Stratified Evaluation** | Fair, aggregate comparison |

### General Advice

1. **Never rely on a single metric.** Report at least one precision, one recall, and one rank-sensitive metric.
2. **Always stratify** by query type to avoid hiding failures behind strong aggregate numbers.
3. **Plot metric@K curves** to find the optimal number of chunks to retrieve.
4. **Use NDCG when possible** — it is the most principled choice for graded relevance.

In [ ]:
# Decision tree visualization
print("Metric Selection Decision Tree")
print("=" * 55)
print()
print("  Is missing a relevant chunk catastrophic?")
print("  ├── YES → Use Recall@K + Context Recall")
print("  └── NO")
print("      │")
print("      Is the relevance binary or graded?")
print("      ├── BINARY → Use MAP + MRR")
print("      └── GRADED → Use NDCG@K + ERR")
print("          │")
print("          Do you need per-category analysis?")
print("          ├── YES → Add Stratified Evaluation")
print("          └── NO  → Report aggregate + P/R curves")
print()
print("  Always also consider:")
print("  • Metric@K curves to find optimal K")
print("  • Chunk Attribution Rate to debug retriever-generator gap")

---
# Summary

In this notebook we covered **17 evaluation metrics** across 5 categories for assessing ranked chunk retrieval in RAG systems:

### Precision-Oriented ("How many retrieved chunks are useful?")
- **P@K** — fraction of top-K that are relevant
- **AP / MAP** — rank-sensitive precision averaged over queries
- **R-Precision** — self-normalizing precision at rank R
- **AUC-PR** — area under the interpolated precision-recall curve

### Recall-Oriented ("How many useful chunks did we find?")
- **R@K / MR@K** — fraction of relevant chunks retrieved
- **Coverage Recall** — fraction of query facets covered

### Rank-Sensitive ("Are the best chunks ranked highest?")
- **DCG / NDCG** — graded relevance with logarithmic rank discount
- **RR / MRR** — speed to first relevant result
- **ERR** — cascading satisfaction model with graded relevance

### RAG-Specific ("Does retrieval quality help generation?")
- **Context Precision** — are relevant chunks ranked first?
- **Context Recall** — are reference claims supported by retrieved context?
- **Hit Rate** — did we find at least one relevant chunk?
- **Chunk Attribution Rate** — are retrieved chunks actually used?

### Evaluation Protocol ("How do we run a rigorous evaluation?")
- **Nugget Recall** — atomic fact coverage
- **Metric@K Curves** — sensitivity analysis
- **Stratified Evaluation** — per-category breakdown